# 🚀 INFERNT — CUDA LLM Inference Engine

A ground-up CUDA/C++ inference engine for LLaMA-family architectures.

**⚠️ IMPORTANT:** Set your runtime to GPU before running: **Runtime → Change runtime type → T4 GPU**

## 1. Verify GPU & CUDA Environment

In [ ]:
!nvidia-smi
!echo ""
!nvcc --version
!echo ""
!cmake --version

## 2. Clone & Build INFERNT

In [ ]:
import subprocess, os

PROJECT_DIR = '/content/INFERNT'
BUILD_DIR   = os.path.join(PROJECT_DIR, 'build')
BINARY      = os.path.join(BUILD_DIR, 'bin', 'infernt')

# Step 1: Clone if not present
if not os.path.exists(PROJECT_DIR):
    !git clone https://github.com/Mahendra1706/INFERNT.git {PROJECT_DIR}
else:
    print(f'Repository already exists at {PROJECT_DIR}')

# Step 2: Detect GPU compute capability for CUDA build
import torch
if torch.cuda.is_available():
    cc_major = torch.cuda.get_device_capability(0)[0]
    cc_minor = torch.cuda.get_device_capability(0)[1]
    cuda_arch = f'{cc_major}{cc_minor}'
    print(f'Detected GPU: {torch.cuda.get_device_name(0)}, Compute Capability: {cc_major}.{cc_minor}')
else:
    cuda_arch = '75'  # Default to T4
    print('WARNING: No CUDA GPU detected, defaulting to compute capability 7.5 (T4)')

# Step 3: Clean build
!rm -rf {BUILD_DIR}
!mkdir -p {BUILD_DIR}

# Step 4: CMake configure (using detected CUDA arch instead of 'native')
cmake_ret = os.system(f'cd {BUILD_DIR} && cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES={cuda_arch}')
if cmake_ret != 0:
    raise RuntimeError('❌ CMake configuration FAILED! Check errors above.')

# Step 5: Build
make_ret = os.system(f'cd {BUILD_DIR} && make -j$(nproc)')
if make_ret != 0:
    raise RuntimeError('❌ Build FAILED! Check errors above.')

# Step 6: Verify binary exists
if os.path.exists(BINARY):
    print(f'\n✅ Build successful! Binary: {BINARY}')
    !ls -lh {BINARY}
else:
    raise RuntimeError(f'❌ Binary not found at {BINARY}. Build failed silently.')

## 3. Run Test Suite (Phases 1-18)

In [ ]:
!{BINARY}

## 4. Convert TinyLlama Checkpoint to INFERNT Format

Downloads TinyLlama-1.1B from Hugging Face and converts it to INFERNT binary format.

In [ ]:
!pip install -q transformers torch accelerate sentencepiece protobuf

!python3 /content/INFERNT/tools/convert_checkpoint.py \
    --model "TinyLlama/TinyLlama-1.1B-Chat-v1.0" \
    --output_dir /content/model_checkpoint

print("\n📁 Checkpoint files:")
!ls -lh /content/model_checkpoint/

## 5. Run Inference with TinyLlama 🧠

Loads the full model onto GPU and generates text autoregressively.

In [ ]:
print("GPU memory BEFORE model load:")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv,noheader
print()

!/content/INFERNT/build/bin/infernt \
    --model /content/model_checkpoint \
    --prompt "Once upon a time, in a high-performance CUDA world," \
    --max-tokens 64 \
    --temp 0.0

print("\nGPU memory AFTER model load:")
!nvidia-smi --query-gpu=memory.used,memory.free --format=csv,noheader

## 6. Try Different Prompts

In [ ]:
!/content/INFERNT/build/bin/infernt \
    --model /content/model_checkpoint \
    --prompt "The future of artificial intelligence" \
    --max-tokens 100 \
    --temp 0.7